# LLIE Colab L4 Pipeline

这份 notebook 面向 Colab `L4 GPU`，用于完成低光照增强论文的第一轮实验复现。

当前主线：

- `URetinex-Net` on `LOL-v1`
- `Diff-Retinex` on `LOL-v1`
- `Reti-Diff` on `LOLv2-Real` and `LOLv2-Synthetic`
- 自动记录状态、日志、指标与汇总表


## 初始化

默认优先使用从 GitHub 克隆下来的仓库代码；Google Drive 只存数据集、权重和运行结果。

如果远端运行环境里还没有 `/content/sspir`，下面的代码会自动克隆仓库。


In [23]:
# 单元 1：同步远端 GitHub 仓库代码
%cd /content/sspir
!git pull


/content/sspir
Already up to date.


In [ ]:
# 单元 2：挂载 Google Drive、定位 notebook 根目录、自动修正资源路径、读取运行配置
from google.colab import drive
from pathlib import Path
import json
import subprocess
import shutil

drive_root = Path('/content/drive')
if not drive_root.exists() or not any(drive_root.iterdir()):
    drive.mount('/content/drive')
else:
    print('Google Drive is already mounted at /content/drive')

REPO_URL = 'https://github.com/yimeng-tong/sspir.git'
REPO_ROOT = Path('/content/sspir')
repo_notebook_root = REPO_ROOT / 'experiments' / 'colab_l4'

if not (repo_notebook_root / 'config' / 'run_config.example.json').exists():
    if REPO_ROOT.exists() and not (REPO_ROOT / '.git').exists():
        raise RuntimeError(
            'Path /content/sspir exists but is not a git repo. Remove it or rename it, then rerun.'
        )
    if not REPO_ROOT.exists():
        print(f'Cloning repository to {REPO_ROOT} ...')
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

candidate_roots = [
    repo_notebook_root,
    Path('/content/drive/MyDrive/thesis/experiments/colab_l4'),
    Path.cwd(),
    Path.cwd() / 'experiments' / 'colab_l4',
]

NOTEBOOK_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'config' / 'run_config.example.json').exists():
        NOTEBOOK_ROOT = candidate
        break

if NOTEBOOK_ROOT is None:
    raise FileNotFoundError(
        'Could not locate experiments/colab_l4. If you cloned the repo, use '
        '/content/sspir/experiments/colab_l4. If you uploaded code to Drive, use '
        '/content/drive/MyDrive/thesis/experiments/colab_l4.'
    )

CONFIG_PATH = NOTEBOOK_ROOT / 'config' / 'run_config.json'
EXAMPLE_CONFIG = NOTEBOOK_ROOT / 'config' / 'run_config.example.json'

if not CONFIG_PATH.exists():
    shutil.copyfile(EXAMPLE_CONFIG, CONFIG_PATH)
    print(f'Created {CONFIG_PATH}. Edit it before continuing.')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

def apply_drive_root(config, asset_root):
    asset_root = Path(asset_root)
    config['drive_root'] = str(asset_root)
    config['output_root'] = str(asset_root / 'runs')
    config['datasets']['lol_v1']['lq_dir'] = str(asset_root / 'datasets' / 'LOL' / 'eval15' / 'low')
    config['datasets']['lol_v1']['gt_dir'] = str(asset_root / 'datasets' / 'LOL' / 'eval15' / 'high')
    config['datasets']['lolv2_real']['lq_dir'] = str(asset_root / 'datasets' / 'LOLv2' / 'Real_captured' / 'Test' / 'Low')
    config['datasets']['lolv2_real']['gt_dir'] = str(asset_root / 'datasets' / 'LOLv2' / 'Real_captured' / 'Test' / 'Normal')
    config['datasets']['lolv2_syn']['lq_dir'] = str(asset_root / 'datasets' / 'LOLv2' / 'Synthetic' / 'Test' / 'Low')
    config['datasets']['lolv2_syn']['gt_dir'] = str(asset_root / 'datasets' / 'LOLv2' / 'Synthetic' / 'Test' / 'Normal')
    config['weights']['diff_retinex']['tdn'] = str(asset_root / 'weights' / 'diff_retinex' / 'checkpoint_LOL_Diff_TDN.pth')
    config['weights']['diff_retinex']['rda'] = str(asset_root / 'weights' / 'diff_retinex' / 'Diff_RDA_best.pth')
    config['weights']['diff_retinex']['ida'] = str(asset_root / 'weights' / 'diff_retinex' / 'Diff_IDA_best.pth')
    config['weights']['reti_diff']['llie_real'] = str(asset_root / 'weights' / 'reti_diff' / 'llie_real.pth')
    config['weights']['reti_diff']['llie_syn'] = str(asset_root / 'weights' / 'reti_diff' / 'llie_syn.pth')
    config['weights']['reti_diff']['retinex_decom'] = str(asset_root / 'weights' / 'reti_diff' / 'retinex_decomnet.pth')

def looks_like_asset_root(candidate):
    candidate = Path(candidate)
    required = [
        candidate / 'datasets' / 'LOL' / 'eval15' / 'low',
        candidate / 'datasets' / 'LOL' / 'eval15' / 'high',
        candidate / 'weights',
    ]
    return all(path.exists() for path in required)

def discover_asset_roots(my_drive):
    candidates = set()
    for low_dir in my_drive.glob('**/datasets/LOL/eval15/low'):
        candidate = low_dir.parents[3]
        if looks_like_asset_root(candidate):
            candidates.add(candidate)
    if candidates:
        return sorted(candidates)

    for candidate in my_drive.rglob('thesis_llie_l4'):
        if candidate.is_dir() and looks_like_asset_root(candidate):
            candidates.add(candidate)
    return sorted(candidates)

configured_drive_root = Path(cfg.get('drive_root', ''))
if not looks_like_asset_root(configured_drive_root):
    my_drive = Path('/content/drive/MyDrive')
    candidates = discover_asset_roots(my_drive)
    if len(candidates) == 1:
        detected_root = candidates[0]
        print(f'Auto-detected drive_root: {detected_root}')
        apply_drive_root(cfg, detected_root)
        CONFIG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    else:
        print(f'Configured drive_root is missing expected assets: {configured_drive_root}')
        print('Detected candidates:')
        for candidate in candidates[:10]:
            print('  ', candidate)

print('Notebook root:', NOTEBOOK_ROOT)
print(json.dumps(cfg, ensure_ascii=False, indent=2))


In [ ]:
# 单元 3：检查 GPU、数据集路径与权重路径是否可用
!nvidia-smi

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

REPOS_ROOT = Path(cfg['repos_root'])
OUTPUT_ROOT = Path(cfg['output_root'])
REPOS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def count_images(path_str):
    path = Path(path_str)
    if not path.exists():
        return -1
    return sum(1 for p in path.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'})

def file_exists(path_str):
    return Path(path_str).exists()

print('Repos root:', REPOS_ROOT)
print('Output root:', OUTPUT_ROOT)
for dataset_name, ds in cfg['datasets'].items():
    low_count = count_images(ds['lq_dir'])
    high_count = count_images(ds['gt_dir'])
    print(f"{dataset_name}: low={low_count} high={high_count}")
    if low_count == -1 or high_count == -1:
        print(f"  low_path={ds['lq_dir']}")
        print(f"  high_path={ds['gt_dir']}")

print('Diff-Retinex weights:')
for name, path_str in cfg['weights']['diff_retinex'].items():
    print(f"  {name}: {file_exists(path_str)} -> {path_str}")

print('Reti-Diff weights:')
for name, path_str in cfg['weights']['reti_diff'].items():
    print(f"  {name}: {file_exists(path_str)} -> {path_str}")


In [ ]:
# 单元 4：安装通用 Python 依赖
!pip -q install pyyaml lpips scikit-image tensorboardX timm einops thop pyiqa openai-clip ftfy addict natsort lmdb facexlib


In [24]:
# 单元 5：克隆基线仓库，并按官方 README 安装 BasicSR 与 Reti-Diff 依赖
import subprocess

repos = {
    'URetinex-Net': 'https://github.com/SZU-AdvTech-2023/262-URetinex-Net-Retinex-based-Deep-Unfolding-Network-for-Low-light-Image-Enhancement.git',
    'Diff-Retinex': 'https://github.com/XunpengYi/Diff-Retinex.git',
    'Reti-Diff': 'https://github.com/ChunmingHe/Reti-Diff.git',
    'BasicSR': 'https://github.com/XPixelGroup/BasicSR.git',
}

for name, url in repos.items():
    target = REPOS_ROOT / name
    if target.exists():
        print(f'Skip clone, already exists: {target}')
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)

subprocess.run(['pip', 'uninstall', '-y', 'basicsr'], check=False)
subprocess.run(['pip', 'install', '-q', 'tb-nightly'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(REPOS_ROOT / 'BasicSR' / 'requirements.txt')], check=True)
subprocess.run(['python', 'setup.py', 'develop'], cwd=str(REPOS_ROOT / 'BasicSR'), check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(REPOS_ROOT / 'Reti-Diff' / 'requirements.txt')], check=True)
subprocess.run(['python', 'setup.py', 'develop'], cwd=str(REPOS_ROOT / 'Reti-Diff'), check=True)

print('Repositories are ready.')


Skip clone, already exists: /content/thesis_llie_repos/URetinex-Net
Skip clone, already exists: /content/thesis_llie_repos/Diff-Retinex
Skip clone, already exists: /content/thesis_llie_repos/Reti-Diff
Skip clone, already exists: /content/thesis_llie_repos/BasicSR
Repositories are ready.


## 运行前检查

下面这个辅助单元只会改 `run_switches`，不会覆盖你的路径配置。

把 `TARGET_PHASE` 改成下面三个值之一：

- `'uretinex'`
- `'diff_retinex'`
- `'reti_diff'`


In [25]:
# 单元 6：切换当前要运行的基线阶段，只改 run_switches，不改路径配置
import json

TARGET_PHASE = 'reti_diff'

switch_map = {
    'uretinex': {
        'uretinex': True,
        'diff_retinex': False,
        'reti_diff': False,
        'compute_metrics': True,
    },
    'diff_retinex': {
        'uretinex': False,
        'diff_retinex': True,
        'reti_diff': False,
        'compute_metrics': True,
    },
    'reti_diff': {
        'uretinex': False,
        'diff_retinex': False,
        'reti_diff': True,
        'compute_metrics': True,
    },
}

if TARGET_PHASE not in switch_map:
    raise ValueError(f'Unsupported TARGET_PHASE: {TARGET_PHASE}')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

cfg['run_switches'] = switch_map[TARGET_PHASE]
CONFIG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')

print('Updated run_switches:')
print(json.dumps(cfg['run_switches'], ensure_ascii=False, indent=2))


Updated run_switches:
{
  "uretinex": false,
  "diff_retinex": false,
  "reti_diff": true,
  "compute_metrics": true
}


In [26]:
# 单元 7：执行当前阶段对应的基线推理；如果失败，自动打印最新一次运行的 status 和 run.log 尾部
import subprocess
import json
from pathlib import Path

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

METHOD_DATASETS = {
    'uretinex': ['lol_v1'],
    'diff-retinex': ['lol_v1'],
    'reti-diff': ['lolv2_real', 'lolv2_syn'],
}

SCRIPT = NOTEBOOK_ROOT / 'scripts' / 'run_baseline.py'

def show_latest_failure(method, dataset_key):
    run_root = OUTPUT_ROOT / method / dataset_key
    if not run_root.exists():
        print(f'No run directory found: {run_root}')
        return
    latest_runs = sorted([p for p in run_root.iterdir() if p.is_dir()], reverse=True)
    if not latest_runs:
        print(f'No run instances found under: {run_root}')
        return
    latest = latest_runs[0]
    print(f'Latest run dir: {latest}')
    status_path = latest / 'status.json'
    log_path = latest / 'run.log'
    if status_path.exists():
        print('----- status.json -----')
        print(status_path.read_text(encoding='utf-8'))
    if log_path.exists():
        print('----- run.log (tail 120 lines) -----')
        log_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(log_lines[-120:]))

for method, dataset_keys in METHOD_DATASETS.items():
    if not cfg['run_switches'].get(method.replace('-', '_'), True):
        print(f'Skip disabled method: {method}')
        continue
    for dataset_key in dataset_keys:
        ds = cfg['datasets'][dataset_key]
        cmd = [
            'python', str(SCRIPT),
            '--method', method,
            '--output-root', str(OUTPUT_ROOT),
            '--dataset-name', dataset_key,
            '--lq-dir', ds['lq_dir'],
            '--gt-dir', ds['gt_dir'],
        ]

        if method == 'uretinex':
            cmd.extend(['--repo-root', str(REPOS_ROOT / 'URetinex-Net')])
        elif method == 'diff-retinex':
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Diff-Retinex'),
                '--diff-tdn-weight', cfg['weights']['diff_retinex']['tdn'],
                '--diff-rda-weight', cfg['weights']['diff_retinex']['rda'],
                '--diff-ida-weight', cfg['weights']['diff_retinex']['ida'],
            ])
        elif method == 'reti-diff':
            template_name = 'test_LLIE_real.yml' if dataset_key == 'lolv2_real' else 'test_LLIE_syn.yml'
            weight_name = 'llie_real' if dataset_key == 'lolv2_real' else 'llie_syn'
            cmd.extend([
                '--repo-root', str(REPOS_ROOT / 'Reti-Diff'),
                '--reti-template', str(REPOS_ROOT / 'Reti-Diff' / 'options' / template_name),
                '--reti-weight', cfg['weights']['reti_diff'][weight_name],
                '--reti-decom-weight', cfg['weights']['reti_diff']['retinex_decom'],
            ])

        print('Running:', ' '.join(cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError:
            show_latest_failure(method, dataset_key)
            raise


Skip disabled method: uretinex
Skip disabled method: diff-retinex
Running: python /content/sspir/experiments/colab_l4/scripts/run_baseline.py --method reti-diff --output-root /content/drive/MyDrive/thesis_llie_l4/runs --dataset-name lolv2_real --lq-dir /content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Low --gt-dir /content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Normal --repo-root /content/thesis_llie_repos/Reti-Diff --reti-template /content/thesis_llie_repos/Reti-Diff/options/test_LLIE_real.yml --reti-weight /content/drive/MyDrive/thesis_llie_l4/weights/reti_diff/llie_real.pth --reti-decom-weight /content/drive/MyDrive/thesis_llie_l4/weights/reti_diff/retinex_decomnet.pth


CalledProcessError: Command '['python', '/content/sspir/experiments/colab_l4/scripts/run_baseline.py', '--method', 'reti-diff', '--output-root', '/content/drive/MyDrive/thesis_llie_l4/runs', '--dataset-name', 'lolv2_real', '--lq-dir', '/content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Low', '--gt-dir', '/content/drive/MyDrive/thesis_llie_l4/datasets/LOLv2/Real_captured/Test/Normal', '--repo-root', '/content/thesis_llie_repos/Reti-Diff', '--reti-template', '/content/thesis_llie_repos/Reti-Diff/options/test_LLIE_real.yml', '--reti-weight', '/content/drive/MyDrive/thesis_llie_l4/weights/reti_diff/llie_real.pth', '--reti-decom-weight', '/content/drive/MyDrive/thesis_llie_l4/weights/reti_diff/retinex_decomnet.pth']' returned non-zero exit status 1.

In [ ]:
# 单元 8：为当前启用的方法计算指标，跳过旧失败记录和空目录
import json
import subprocess
from pathlib import Path

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

enabled_methods = {
    method.replace('_', '-'): enabled
    for method, enabled in cfg['run_switches'].items()
    if method != 'compute_metrics' and enabled
}

metrics_script = NOTEBOOK_ROOT / 'scripts' / 'compute_metrics.py'
meta_files = sorted(OUTPUT_ROOT.rglob('run_metadata.json'))

for meta_file in meta_files:
    meta = json.loads(meta_file.read_text(encoding='utf-8'))
    method = meta['method']
    if method not in enabled_methods:
        print(f'Skip disabled method metadata: {meta_file}')
        continue

    status_path = Path(meta['status_path']) if meta.get('status_path') else None
    if status_path and status_path.exists():
        run_status = json.loads(status_path.read_text(encoding='utf-8'))
        if run_status.get('state') != 'completed':
            print(f'Skip incomplete run: {meta_file}')
            continue

    pred_dir = Path(meta['pred_dir'])
    if not pred_dir.exists():
        print(f'Skip missing prediction dir: {pred_dir}')
        continue

    pred_images = [
        p for p in pred_dir.rglob('*')
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}
    ]
    if not pred_images:
        print(f'Skip empty prediction dir: {pred_dir}')
        continue

    run_dir = meta_file.parent
    metrics_dir = run_dir / 'metrics'
    cmd = [
        'python', str(metrics_script),
        '--method', method,
        '--dataset-name', meta['dataset_name'],
        '--pred-dir', str(pred_dir),
        '--gt-dir', meta['gt_dir'],
        '--output-dir', str(metrics_dir),
    ]
    if meta.get('status_path'):
        cmd.extend(['--status-path', meta['status_path']])
    if method == 'uretinex':
        cmd.extend(['--strip-suffix', '_URetinexNet'])
    cmd.extend(['--skip-empty', '--skip-no-match'])
    if cfg['run_switches'].get('compute_metrics', True):
        print('Computing metrics:', ' '.join(cmd))
        subprocess.run(cmd, check=True)


In [ ]:
# 单元 9：汇总当前启用方法的结果，生成 markdown/csv 表格
import json
import subprocess

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = json.load(f)

enabled_methods = [
    method.replace('_', '-')
    for method, enabled in cfg['run_switches'].items()
    if method != 'compute_metrics' and enabled
]

aggregate_script = NOTEBOOK_ROOT / 'scripts' / 'aggregate_results.py'
summary_root = OUTPUT_ROOT
aggregate_root = OUTPUT_ROOT / 'aggregated'
cmd = [
    'python', str(aggregate_script),
    '--input-root', str(summary_root),
    '--output-root', str(aggregate_root),
    '--only-completed',
]
for method in enabled_methods:
    cmd.extend(['--allowed-method', method])

subprocess.run(cmd, check=True)
md_path = aggregate_root / 'aggregated_metrics.md'
if md_path.exists():
    print(md_path.read_text(encoding='utf-8'))
else:
    print('No aggregated markdown was generated yet.')


## 后续建议

1. `Diff-Retinex` 跑完后，再把 `TARGET_PHASE` 切到 `reti_diff`。
2. 每一轮跑完都保留 `status.json`、`run.log`、`metrics/summary.json`。
3. 最后把各方法均值指标和代表图像整理到第四章主表与定性对比图。
